# EDA v2 — cupons iFood

Exploração inicial do case iFood com uma separação deliberada de responsabilidades: Spark carrega e reduz dados; pandas analisa, tabula e prepara visualizações.

```mermaid
flowchart LR
  subgraph spark [Spark]
    RawJSON[raw JSONs]
    Processed[processed parquet]
    Agg[groupBy / count / filter]
    RawJSON --> Agg
    Processed --> Agg
  end
  subgraph pandas [Pandas]
    Small[DataFrames pequenos]
    Tables[tabelas e stats]
    Plots[figuras Plotly]
    Small --> Tables
    Small --> Plots
  end
  Agg -->|toPandas| Small
```

Figuras usam primitivas e tema de `src/viz.py` (skill statistical-plots); o notebook só agrega e exibe.

## 1. Setup

Fonte: raw e processed. Inicializa caminhos, configuração e DataFrames Spark, deixando a carga e o cache como responsabilidade do Spark.

In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import pandas as pd

from src.clean import normalize_profile
from src.config import load
from src.io import parse_events, read_offers, read_profile
from src.pipeline import build_spark

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

cfg = load()
spark = build_spark(cfg, app_name="eda-v2")
spark.sparkContext.setLogLevel("ERROR")

events = parse_events(spark, cfg).cache()
offers = read_offers(spark, cfg).cache()
raw_profile = read_profile(spark, cfg).cache()
profile = normalize_profile(raw_profile, cfg).cache()
processed = spark.read.parquet(str(cfg.processed_dir)).cache()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/12 11:29:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/07/12 11:29:02 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


## 2. Panorama dos dados

Fonte: raw e processed. Mede a volumetria das fontes e do grão analítico antes de interpretar qualquer taxa.

In [6]:
from pyspark.sql import functions as F

from src import viz
from src.viz import (
    format_pt_br,
    horizontal_bars,
    setup_theme,
    vertical_share_bars,
)

# SPARK REDUCE
n_linhas = processed.count()
n_clientes = processed.select("account_id").distinct().count()
n_ofertas_processed = processed.select("offer_id").distinct().count()
n_events = events.count()

eventos_por_tipo = (
    events.groupBy("event")
    .agg(F.count("*").alias("eventos"), F.min("time").alias("primeiro_dia"))
    .toPandas()
    .sort_values("eventos", ascending=False)
    .reset_index(drop=True)
)

volumetria = pd.DataFrame([
    {
        "fonte": "events (bruto)",
        "linhas": n_events,
        "clientes": events.select("account_id").distinct().count(),
        "ofertas": pd.NA,
        "grao": "eventos de 4 tipos",
    },
    {
        "fonte": "profile (bruto)",
        "linhas": profile.count(),
        "clientes": profile.select("account_id").distinct().count(),
        "ofertas": pd.NA,
        "grao": "1 linha por cliente",
    },
    {
        "fonte": "offers (bruto)",
        "linhas": offers.count(),
        "clientes": pd.NA,
        "ofertas": offers.select("id").distinct().count(),
        "grao": "catálogo de ofertas",
    },
    {
        "fonte": "processed",
        "linhas": n_linhas,
        "clientes": n_clientes,
        "ofertas": n_ofertas_processed,
        "grao": "(account_id, offer_id, received_time)",
    },
])

# PANDAS ANALYSIS
eventos_por_tipo["fracao"] = eventos_por_tipo["eventos"] / eventos_por_tipo["eventos"].sum()

display(volumetria)
display(eventos_por_tipo)

ratio = n_linhas / n_events
print(
    f"O grão processado condensa {format_pt_br(n_events)} eventos brutos em "
    f"{format_pt_br(n_linhas)} recebimentos ({ratio:.1%} do volume de eventos), "
    f"cobrindo {format_pt_br(n_clientes)} clientes e as {n_ofertas_processed} ofertas do catálogo."
)

setup_theme()

horizontal_bars(
    volumetria,
    category="fonte",
    value="linhas",
    title="O bruto é grande; o grão processado condensa o que importa",
    subtitle=f"events: {format_pt_br(n_events)} · processed: {format_pt_br(n_linhas)} · escala log",
    xlabel="Linhas",
    log_scale=False,
).show()

horizontal_bars(
    volumetria.dropna(subset=["clientes"]),
    category="fonte",
    value="clientes",
    title="Quase todos os clientes aparecem nas três fontes",
    subtitle=f"Grão processado cobre {format_pt_br(n_clientes)} clientes",
    xlabel="Clientes",
    color=viz.palette()[1],
).show()

vertical_share_bars(
    eventos_por_tipo,
    category="event",
    value="eventos",
    share="fracao",
    title="Transações dominam o fluxo bruto de eventos",
    subtitle=f"n = {format_pt_br(n_events)} eventos",
).show()

,fonte,linhas,clientes,ofertas,grao
0,events (bruto),306534,17000,<NA>,eventos de 4 tipos
1,profile (bruto),17000,17000,<NA>,1 linha por cliente
2,offers (bruto),10,<NA>,10,catálogo de ofertas
3,processed,76277,16994,10,"(account_id, offer_id, received_time)"


,event,eventos,primeiro_dia,fracao
0,transaction,138953,0.0,0.453304
1,offer received,76277,0.0,0.248837
2,offer viewed,57725,0.0,0.188315
3,offer completed,33579,0.0,0.109544


O grão processado condensa 306.534 eventos brutos em 76.277 recebimentos (24.9% do volume de eventos), cobrindo 16.994 clientes e as 10 ofertas do catálogo.


## 3. Catálogo de ofertas

Fonte: raw. Resume os tipos, descontos, mínimos, duração e canais das ofertas disponíveis para entender o espaço de decisão.

In [3]:
from src import viz
from src.viz import format_pt_br, grouped_bars, horizontal_bars

# SPARK REDUCE
catalogo = (
    offers.select("id", "offer_type", "min_value", "duration", "discount_value", "channels")
    .toPandas()
    .sort_values(["offer_type", "discount_value", "min_value"])
    .reset_index(drop=True)
)

por_tipo = (
    catalogo.groupby("offer_type", as_index=False)
    .agg(
        ofertas=("id", "count"),
        min_medio=("min_value", "mean"),
        desconto_medio=("discount_value", "mean"),
        duracao_media=("duration", "mean"),
    )
)

# PANDAS ANALYSIS
catalogo_plot = catalogo.assign(
    rotulo=lambda d: (
        d["offer_type"].str[:4]
        + " · desc "
        + d["discount_value"].astype(int).astype(str)
        + " · min "
        + d["min_value"].astype(int).astype(str)
    ),
)

display(catalogo)
display(por_tipo.round(1))

print(
    f"{len(catalogo)} ofertas, {catalogo['offer_type'].nunique()} tipos — "
    f"informational tem desconto e mínimo zero; duração varia de "
    f"{int(catalogo['duration'].min())} a {int(catalogo['duration'].max())} dias."
)

horizontal_bars(
    por_tipo,
    category="offer_type",
    value="ofertas",
    title="O catálogo é pequeno e desbalanceado por tipo",
    subtitle="4 bogo · 4 discount · 2 informational",
    xlabel="Ofertas no catálogo",
    color=viz.palette()[0],
).show()

grouped_bars(
    catalogo_plot.sort_values(["offer_type", "discount_value"]),
    category="rotulo",
    series=["min_value", "discount_value", "duration"],
    series_names=["Gasto mínimo", "Desconto", "Duração (dias)"],
    orientation="h",
    title="Cada oferta combina regra de desconto, mínimo e janela própria",
    subtitle="informational: recompensa zero · validade não é constante no pipeline",
    xlabel="Valor / dias",
    value_label="%{x:.0f}",
).show()

,id,offer_type,min_value,duration,discount_value,channels
0,9b98b8c7a33c4b65b9aebfe6a799e6d9,bogo,5,7.0,5,"[web, email, mobile]"
1,f19421c1d4aa40978ebb69ca19b0e20d,bogo,5,5.0,5,"[web, email, mobile, social]"
2,ae264e3637204a6fb9bb56bc8210ddfd,bogo,10,7.0,10,"[email, mobile, social]"
3,4d5c57ea9a6940dd891ad53e9dbe8da0,bogo,10,5.0,10,"[web, email, mobile, social]"
4,fafdcd668e3743c1bb461111dcafc2a4,discount,10,10.0,2,"[web, email, mobile, social]"
5,2906b810c7d4411798c6938adc9daaa5,discount,10,7.0,2,"[web, email, mobile]"
6,2298d6c36e964ae4a3e7e9706d1fb8c2,discount,7,7.0,3,"[web, email, mobile, social]"
7,0b1e1539f2cc45b7b9fa7c272da2e1d7,discount,20,10.0,5,"[web, email]"
8,3f207df678b143eea3cee63160fa8bed,informational,0,4.0,0,"[web, email, mobile]"
9,5a8bc65990b245e5a138643cd4eb9837,informational,0,3.0,0,"[email, mobile, social]"


,offer_type,ofertas,min_medio,desconto_medio,duracao_media
0,bogo,4,7.5,7.5,6.0
1,discount,4,11.8,3.0,8.5
2,informational,2,0.0,0.0,3.5


10 ofertas, 3 tipos — informational tem desconto e mínimo zero; duração varia de 3 a 10 dias.


## 4. Eventos no tempo

Fonte: raw. Verifica quando cada tipo de evento acontece e se os envios são ondas discretas ou atividade contínua.

In [4]:
from src import eda
from src.viz import faceted, format_pt_br, timeline_ranges

# SPARK REDUCE
por_dia = eda.events_over_time(events)
ondas = sorted(por_dia.loc[por_dia["event"] == "offer received", "dia"].unique())
dia_max = int(por_dia["dia"].max())
janelas, fim_do_teste = eda.campaign_validity_windows(processed, events)

totais = (
    por_dia.groupby("event", observed=False)["eventos"]
    .sum()
    .reindex(eda.EVENT_ORDER)
    .reset_index()
)
n_censurados = int(janelas["recebimentos_censurados"].sum())
n_linhas = processed.count()

# PANDAS ANALYSIS
display(totais)
display(janelas.round(3))
print(
    f"Disparos em {len(ondas)} dias (t={', '.join(str(int(d)) for d in ondas)}); "
    f"fim dos dados em t={fim_do_teste:g}."
)

rotulos = {
    "offer received": "recebimentos",
    "offer viewed": "views por dia",
    "offer completed": "completados por dia",
    "transaction": "compras por dia",
}
paineis = []
for evento in eda.EVENT_ORDER:
    sub = por_dia.loc[por_dia["event"] == evento].sort_values("dia")
    paineis.append({
        "title": rotulos[evento],
        "x": sub["dia"].tolist(),
        "y": sub["eventos"].tolist(),
        "kind": "line" if evento == "transaction" else "bar",
    })

faceted(
    paineis,
    title="Seis disparos de campanha; resposta decai entre ondas",
    subtitle=(
        f"Ondas em t={', '.join(str(int(d)) for d in ondas)} · "
        "escala linear por painel · pontilhado = disparo"
    ),
    cols=2,
    kind="bar",
    vlines=ondas,
    xlabel="dia desde o início do teste",
    x_range=(-0.5, dia_max + 0.5),
    x_dtick=5,
    y_tickformat=",.0f",
    row_height=220,
).show()

timeline_ranges(
    janelas,
    label="rotulo",
    start="received_time",
    end="valid_until",
    observed_end="janela_observavel",
    censored="censurada",
    title="Ondas tardias têm validade cortada pelo fim da coleta",
    subtitle=(
        f"{format_pt_br(n_censurados)} recebimentos ({n_censurados / n_linhas:.1%}) "
        "com janela além de t=" + f"{fim_do_teste:g}"
    ),
    xlabel="dia desde o início do teste",
    vline=fim_do_teste,
).show()

print(
    f"Censura à direita: {format_pt_br(n_censurados)} recebimentos "
    f"({n_censurados / n_linhas:.1%}) não observam a validade inteira — "
    "conversão das ondas 4–5 fica subestimada."
)

,event,eventos
0,offer received,76277
1,offer viewed,57725
2,offer completed,33579
3,transaction,138953


,campaign_wave,received_time,duration,recebimentos,valid_until,janela_observavel,censurada,dias_censurados,rotulo,recebimentos_censurados
0,0,0.0,6.526,12650,6.526,6.526,False,0.000,onda 0 · t=0 · 6.52569d,0
1,1,7.0,6.495,12669,13.495,13.495,False,0.000,onda 1 · t=7 · 6.49491d,0
2,2,14.0,6.511,12711,20.511,20.511,False,0.000,onda 2 · t=14 · 6.51066d,0
3,3,17.0,6.480,12778,23.480,23.480,False,0.000,onda 3 · t=17 · 6.48036d,0
4,4,21.0,6.508,12704,27.508,27.508,False,0.000,onda 4 · t=21 · 6.50834d,0
5,5,24.0,6.502,12765,30.502,29.750,True,0.752,onda 5 · t=24 · 6.50247d,12765


Disparos em 6 dias (t=0, 7, 14, 17, 21, 24); fim dos dados em t=29.75.


Censura à direita: 12.765 recebimentos (16.7%) não observam a validade inteira — conversão das ondas 4–5 fica subestimada.


## 5. Qualidade dos dados brutos

Fonte: raw. Procura problemas visíveis antes do pipeline: referências ausentes, eventos fora de ordem, perfil faltante e sentinelas.

In [5]:
from pyspark.sql import functions as F

from src import eda
from src.attribution import attribute
from src.viz import format_pt_br, vertical_bars

# SPARK REDUCE
n_txn = events.filter(F.col("event") == "transaction").count()
n_txn_ref = events.filter(
    (F.col("event") == "transaction") & F.col("offer_ref").isNotNull()
).count()
n_camp = events.filter(F.col("event") != "transaction").count()
n_camp_ref = events.filter(
    (F.col("event") != "transaction") & F.col("offer_ref").isNotNull()
).count()
dup_events = (
    events.groupBy("account_id", "event", "time", "offer_ref")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

sem_view = eda.completed_unseen_by_type(events, offers)
nulos_perfil = eda.identity_null_overlap(raw_profile, cfg)
attributed = attribute(events, offers, cfg).cache()
fora_janela = eda.unattributable_transaction_share(events, attributed)
pago_abaixo = eda.paid_below_minimum(processed)

com_evento = sem_view[sem_view["completados"] > 0]
taxa_sem_view = com_evento["sem_view"].sum() / com_evento["completados"].sum()

# PANDAS ANALYSIS
qualidade = pd.DataFrame([
    {
        "métrica": "transações sem offer_ref",
        "numerador": n_txn - n_txn_ref,
        "denominador": n_txn,
        "taxa": (n_txn - n_txn_ref) / n_txn,
        "leitura": "esperado — ligação oferta→compra é por atribuição temporal",
    },
    {
        "métrica": "eventos de campanha sem offer_ref",
        "numerador": n_camp - n_camp_ref,
        "denominador": n_camp,
        "taxa": (n_camp - n_camp_ref) / n_camp,
        "leitura": "problema se > 0",
    },
    {
        "métrica": "offer completed sem view prévia",
        "numerador": int(com_evento["sem_view"].sum()),
        "denominador": int(com_evento["completados"].sum()),
        "taxa": taxa_sem_view,
        "leitura": "esperado — controle converte sem exposição (G3)",
    },
    {
        "métrica": "transações fora de qualquer janela de oferta",
        "numerador": int(fora_janela.loc[fora_janela["grupo"].str.startswith("fora"), "transacoes"].iloc[0]),
        "denominador": int(fora_janela["transacoes"].sum()),
        "taxa": float(fora_janela.loc[fora_janela["grupo"].str.startswith("fora"), "fracao"].iloc[0]),
        "leitura": "comportamento espontâneo — não atribuível a envio",
    },
    {
        "métrica": "perfil com age sentinela (118)",
        "numerador": int(nulos_perfil.loc[nulos_perfil["conjunto"] == f"age={cfg.age_sentinel}", "clientes"].iloc[0]),
        "denominador": int(nulos_perfil["total_clientes"].iloc[0]),
        "taxa": float(nulos_perfil.loc[nulos_perfil["conjunto"] == f"age={cfg.age_sentinel}", "fracao"].iloc[0]),
        "leitura": "esperado — vira identity_missing no pipeline",
    },
    {
        "métrica": "linhas brutas duplicadas (account, event, time, offer_ref)",
        "numerador": dup_events,
        "denominador": events.count(),
        "taxa": dup_events / events.count(),
        "leitura": "problema se > 0",
    },
])

display(qualidade.assign(taxa=lambda d: d["taxa"].map("{:.1%}".format)))
display(nulos_perfil)
display(pago_abaixo)

vertical_bars(
    sem_view[sem_view["completados"] > 0],
    category="offer_type",
    value="taxa_sem_view",
    title="Um quarto dos completados nunca foi visto — controle real no funil",
    subtitle=f"taxa geral {taxa_sem_view:.1%} sobre completados",
    ylabel="Taxa sem view prévia",
    label_template="%{y:.1%}",
    tickformat=".0%",
    hovertemplate="%{x}<br>%{y:.1%}<extra></extra>",
).show()

print(
    f"Transações com offer_ref: {n_txn_ref} de {format_pt_br(n_txn)} — "
    "nenhuma; a ligação oferta→compra só existe por atribuição temporal."
)
print(
    f"Conversões pagas abaixo do mínimo (auditoria G10): "
    f"{int(pago_abaixo['abaixo_do_minimo'].sum())} — deve ser zero."
)

Premissa 1 violada em 15721 transação(ões): mais de uma oferta ativa no intervalo; prioridade 'earliest_received' aplicada.


,métrica,numerador,denominador,taxa,leitura
0,transações sem offer_ref,138953,138953,100.0%,esperado — ligação oferta→compra é por atribui...
1,eventos de campanha sem offer_ref,0,167581,0.0%,problema se > 0
2,offer completed sem view prévia,8568,33182,25.8%,esperado — controle converte sem exposição (G3)
3,transações fora de qualquer janela de oferta,17090,138953,12.3%,comportamento espontâneo — não atribuível a envio
4,perfil com age sentinela (118),2175,17000,12.8%,esperado — vira identity_missing no pipeline
5,"linhas brutas duplicadas (account, event, time...",396,306534,0.1%,problema se > 0


,conjunto,clientes,fracao,total_clientes
0,age=118,2175,0.127941,17000
1,gender nulo,2175,0.127941,17000
2,credit_card_limit nulo,2175,0.127941,17000
3,"os três, juntos",2175,0.127941,17000
4,ao menos um,2175,0.127941,17000


,offer_type,conversoes_pagas,abaixo_do_minimo,custo_acima_da_receita,custo_total,custo_sob_suspeita,frac_abaixo_do_minimo,frac_custo_sob_suspeita
0,bogo,13496,0,0,96795.0,0.0,0.0,0.0
1,discount,12500,0,0,34773.0,0.0,0.0,0.0


Transações com offer_ref: 0 de 138.953 — nenhuma; a ligação oferta→compra só existe por atribuição temporal.
Conversões pagas abaixo do mínimo (auditoria G10): 0 — deve ser zero.


## 6. Grão processado

Fonte: processed. Confirma a unidade analítica do dataset preparado e identifica nulos, duplicatas ou colunas que exigem cuidado.

In [6]:
from pyspark.sql import functions as F

from src import eda
from src.contract import CONTRACT_COLUMNS, NULLABLE_COLUMNS

# SPARK REDUCE
n_linhas = processed.count()
dupes = (
    processed.groupBy("account_id", "offer_id", "received_time")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
nulos_row = processed.select(
    *[F.sum(F.col(c).isNull().cast("long")).alias(c) for c in CONTRACT_COLUMNS]
).first().asDict()
contagens = (
    processed.groupBy("offer_type", "treatment", "converted", "campaign_wave")
    .agg(F.count("*").alias("linhas"))
    .orderBy("campaign_wave", "offer_type", "treatment", "converted")
    .toPandas()
)
amostra = (
    processed.orderBy("campaign_wave", "account_id", "offer_id")
    .limit(5)
    .toPandas()
)

# PANDAS ANALYSIS
nulos = pd.DataFrame(
    [{"coluna": c, "nulos": int(nulos_row[c] or 0)} for c in CONTRACT_COLUMNS if (nulos_row[c] or 0) > 0]
)
checks = eda.sanity_checks(processed)

display(pd.DataFrame([
    {"verificação": "grão único (account_id, offer_id, received_time)", "valor": dupes, "esperado": 0},
    {"verificação": "linhas no grão", "valor": n_linhas, "esperado": "76.277"},
]))
display(nulos if len(nulos) else pd.DataFrame({"coluna": ["—"], "nulos": [0]}))
display(contagens.head(12))
display(checks)
display(amostra)

print(
    "Unidade analítica: uma linha por recebimento de oferta "
    "(account_id, offer_id, received_time), com label e features as-of esse instante."
)

,verificação,valor,esperado
0,"grão único (account_id, offer_id, received_time)",0,0
1,linhas no grão,76277,76.277


,coluna,nulos
0,age,9776
1,credit_card_limit,9776
2,hist_recency_days,20952
3,hist_time_view_to_conv,47850


,offer_type,treatment,converted,campaign_wave,linhas
0,bogo,0,0,0,478
1,bogo,0,1,0,230
2,bogo,1,0,0,2264
3,bogo,1,1,0,2046
4,discount,0,0,0,1073
5,discount,0,1,0,292
6,discount,1,0,0,1851
7,discount,1,1,0,1877
8,informational,0,0,0,541
9,informational,0,1,0,318


,verificação,linhas,fracao
0,converted=1 com conversion_value = 0,0,0.0
1,conversion_value > 0 sem converted,0,0.0
2,reward_cost > 0 sem conversão (viola G6),0,0.0
3,reward_cost > 0 em informational (viola G6),0,0.0
4,reward_cost acima da receita da conversão,0,0.0
5,ticket médio histórico sem transação histórica,0,0.0
6,gasto histórico negativo,0,0.0
7,"taxa de view histórica fora de [0,1]",0,0.0
8,idade igual à sentinela (viola G7),0,0.0


,account_id,offer_id,offer_type,received_time,campaign_wave,treatment,converted,conversion_value,reward_cost,is_recurrent,age,gender,credit_card_limit,identity_missing,tenure_days,hist_spend_total,hist_txn_count,hist_avg_ticket,hist_spend_std,hist_recency_days,hist_frequency,hist_spend_trend,hist_offers_received,hist_offers_received_bogo,hist_offers_received_discount,hist_offers_received_info,hist_offers_viewed,hist_offers_completed,hist_view_rate,hist_conv_rate_bogo,hist_conv_rate_discount,hist_completed_unseen_flag,hist_time_view_to_conv,discount_value,min_value,duration,n_channels,channel_web,channel_email,channel_mobile,channel_social,discount_to_minvalue_ratio,n_concurrent_offers
0,0011e0d4e6b944f998e987f904e8c1e5,3f207df678b143eea3cee63160fa8bed,informational,0.0,0,1,0,0.00,0.0,0,40,O,57000.0,0,198,0.0,0,0.0,0.0,NaN,0.0,0.0,0,0,0,0,0,0,0.0,0.0,0.0,0,NaN,0.0,0.0,4.0,3,1,1,1,0,0.0,0
1,0020c2b971eb4e9188eac86d93036a77,fafdcd668e3743c1bb461111dcafc2a4,discount,0.0,0,1,1,17.63,2.0,0,59,F,90000.0,0,874,0.0,0,0.0,0.0,NaN,0.0,0.0,0,0,0,0,0,0,0.0,0.0,0.0,0,NaN,2.0,10.0,10.0,4,1,1,1,1,0.2,0
2,003d66b6608740288d6cc97a6903f4f0,5a8bc65990b245e5a138643cd4eb9837,informational,0.0,0,1,1,0.44,0.0,0,26,F,73000.0,0,400,0.0,0,0.0,0.0,NaN,0.0,0.0,0,0,0,0,0,0,0.0,0.0,0.0,0,NaN,0.0,0.0,3.0,3,0,1,1,1,0.0,0
3,00426fe3ffde4c6b9cb9ad6d077a13ea,5a8bc65990b245e5a138643cd4eb9837,informational,0.0,0,1,1,5.33,0.0,0,19,F,65000.0,0,716,0.0,0,0.0,0.0,NaN,0.0,0.0,0,0,0,0,0,0,0.0,0.0,0.0,0,NaN,0.0,0.0,3.0,3,0,1,1,1,0.0,0
4,005500a7188546ff8a767329a2f7c76a,ae264e3637204a6fb9bb56bc8210ddfd,bogo,0.0,0,1,0,0.00,0.0,0,56,M,47000.0,0,229,0.0,0,0.0,0.0,NaN,0.0,0.0,0,0,0,0,0,0,0.0,0.0,0.0,0,NaN,10.0,10.0,7.0,3,0,1,1,1,1.0,0


Unidade analítica: uma linha por recebimento de oferta (account_id, offer_id, received_time), com label e features as-of esse instante.


## 7. Perfil e features do cliente

Fonte: raw e processed. Descreve cadastro, features históricas criadas pelo pipeline (`hist_*`) e exposição observada no teste.

In [7]:
import numpy as np

from pyspark.sql import functions as F

from src import eda
from src.viz import faceted, format_pt_br, heatmap

# SPARK REDUCE — perfil de cadastro
perfil_num = eda.numeric_profile(profile, ["age", "credit_card_limit", "tenure_days"], cfg)
perfil_cat = eda.categorical_profile(profile, ["gender"])
clientes = eda.client_features(processed, events)

n_profile = profile.select("account_id").distinct().count()
n_processed = processed.select("account_id").distinct().count()
n_sentinela = profile.filter(F.col("identity_missing") == 1).count()

hist_compra = eda.numeric_profile(processed, list(eda.HIST_COMPRA_FEATURES), cfg)
hist_oferta = eda.numeric_profile(processed, list(eda.HIST_OFERTA_FEATURES), cfg)
corr_hist = eda.correlation_matrix(
    processed,
    ["hist_spend_total", "hist_txn_count", "hist_avg_ticket", "hist_frequency", "hist_view_rate"],
)
redundantes = eda.redundant_pairs(corr_hist, cfg)

cobertura = pd.DataFrame([
    {"fonte": "profile", "clientes": n_profile},
    {"fonte": "processed", "clientes": n_processed},
    {"fonte": "identity_missing=1", "clientes": n_sentinela},
])
exposicao = clientes[["offers_received", "offers_viewed", "view_rate", "conv_rate"]].describe()

# PANDAS ANALYSIS
display(cobertura)
display(perfil_num.round(3))
display(perfil_cat)
display(hist_compra.round(3))
display(hist_oferta.round(3))
display(redundantes if len(redundantes) else pd.DataFrame({"feature_a": [], "feature_b": [], "r": []}))
display(exposicao.round(3))

panels_perfil = []
for coluna, frame in [
    ("credit_card_limit", profile),
    ("tenure_days", profile),
    ("spend_total", None),
]:
    if frame is not None:
        hist = eda.numeric_histogram(frame, coluna, cfg.histogram_bins)
    else:
        hist = (
            clientes[["spend_total"]]
            .assign(bucket=pd.cut(clientes["spend_total"], bins=cfg.histogram_bins))
            .groupby("bucket", observed=False)
            .size()
            .reset_index(name="contagem")
        )
        hist["centro"] = hist["bucket"].apply(lambda b: b.mid if pd.notna(b) else np.nan)
        hist = hist.dropna(subset=["centro"])
    panels_perfil.append({"title": coluna, "x": hist["centro"].tolist(), "y": hist["contagem"].tolist()})

faceted(
    panels_perfil,
    title="Cadastro e gasto observado no teste",
    subtitle=f"{format_pt_br(n_profile)} clientes · {format_pt_br(n_sentinela)} com identidade ausente",
    kind="bar",
    cols=3,
    xlabel="valor",
).show()

panels_hist = []
for coluna in ["hist_spend_total", "hist_view_rate", "hist_offers_received_discount"]:
    hist = eda.numeric_histogram(processed, coluna, cfg.histogram_bins)
    panels_hist.append({"title": coluna, "x": hist["centro"].tolist(), "y": hist["contagem"].tolist()})

faceted(
    panels_hist,
    title="Features históricas carregam cauda de gasto e hábito de resposta",
    subtitle="calculadas as-of received_time · zeros legítimos onde não há histórico",
    kind="bar",
    cols=3,
    xlabel="valor",
).show()

heatmap(
    corr_hist,
    title="Histórico de compra e view rate não são ortogonais",
    subtitle=f"|r| ≥ {cfg.correlation_threshold} sinaliza redundância",
    diverging=True,
).show()

print(
    f"O processed cobre {format_pt_br(n_processed)} de {format_pt_br(n_profile)} clientes; "
    f"as `hist_*` resumem comportamento pré-envio por linha de recebimento."
)

,fonte,clientes
0,profile,17000
1,processed,16994
2,identity_missing=1,2175


,coluna,n,nulos,frac_nulos,zeros,frac_zeros,media,desvio,min,p1,p25,p50,p75,p99,max,outliers,frac_outliers
0,age,14825,2175,0.128,0,0.000,54.394,17.384,18.0,19.0,42.0,55.0,66.0,92.0,101.0,0,0.000
1,credit_card_limit,14825,2175,0.128,0,0.000,65404.992,21598.299,30000.0,31000.0,49000.0,64000.0,80000.0,116000.0,120000.0,0,0.000
2,tenure_days,17000,0,0.000,22,0.001,517.450,411.224,0.0,8.0,208.0,358.0,789.0,1727.0,1823.0,295,0.017


,nivel,linhas,coluna,fracao
0,M,8484,gender,0.499059
1,F,6129,gender,0.360529
2,unknown,2175,gender,0.127941
3,O,212,gender,0.012471


,coluna,n,nulos,frac_nulos,zeros,frac_zeros,media,desvio,min,p1,p25,p50,p75,p99,max,outliers,frac_outliers
0,hist_spend_total,76277,0,0.000,20952,0.275,43.816,78.499,0.00,0.00,0.000,16.140,59.770,276.290,1325.720,4537,0.059
1,hist_txn_count,76277,0,0.000,20952,0.275,3.457,3.667,0.00,0.00,0.000,3.000,5.000,15.000,29.000,2214,0.029
2,hist_avg_ticket,76277,0,0.000,20952,0.275,9.763,17.992,0.00,0.00,0.000,3.778,16.718,37.350,962.100,653,0.009
3,hist_spend_std,76277,0,0.000,29062,0.381,4.436,22.427,0.00,0.00,0.000,1.611,4.296,37.288,679.657,2323,0.030
4,hist_recency_days,55325,20952,0.275,0,0.000,3.354,3.083,0.25,0.25,1.000,2.500,4.750,13.750,24.000,2234,0.040
5,hist_frequency,76277,0,0.000,20952,0.275,0.203,0.202,0.00,0.00,0.000,0.143,0.294,0.857,2.000,1412,0.019
6,hist_spend_trend,76277,0,0.000,20961,0.275,0.885,24.074,-962.10,-28.63,-1.225,0.000,2.197,31.030,961.180,16189,0.212


,coluna,n,nulos,frac_nulos,zeros,frac_zeros,media,desvio,min,p1,p25,p50,p75,p99,max,outliers,frac_outliers
0,hist_offers_received,76277,0,0.000,16994,0.223,1.872,1.454,0.0,0.0,1.000,2.00,3.000,5.0,5.00,0,0.000
1,hist_offers_viewed,76277,0,0.000,21991,0.288,1.425,1.252,0.0,0.0,0.000,1.00,2.000,5.0,5.00,0,0.000
2,hist_offers_completed,76277,0,0.000,43057,0.564,0.738,1.028,0.0,0.0,0.000,0.00,1.000,4.0,5.00,5871,0.077
3,hist_offers_received_bogo,76277,0,0.000,37434,0.491,0.747,0.886,0.0,0.0,0.000,1.00,1.000,3.0,5.00,3474,0.046
4,hist_offers_received_discount,76277,0,0.000,37356,0.490,0.751,0.891,0.0,0.0,0.000,1.00,1.000,3.0,5.00,3509,0.046
5,hist_view_rate,76277,0,0.000,21991,0.288,0.598,0.426,0.0,0.0,0.000,0.75,1.000,1.0,1.00,0,0.000
6,hist_conv_rate_bogo,76277,0,0.000,56006,0.734,0.238,0.409,0.0,0.0,0.000,0.00,0.500,1.0,1.00,0,0.000
7,hist_conv_rate_discount,76277,0,0.000,54131,0.710,0.261,0.423,0.0,0.0,0.000,0.00,0.500,1.0,1.00,0,0.000
8,hist_completed_unseen_flag,76277,0,0.000,64682,0.848,0.152,0.359,0.0,0.0,0.000,0.00,0.000,1.0,1.00,11595,0.152
9,hist_time_view_to_conv,28427,47850,0.627,2063,0.073,2.003,1.872,0.0,0.0,0.625,1.50,2.875,8.0,22.25,901,0.032


,feature_a,feature_b,r,abs_r
0,hist_txn_count,hist_frequency,0.881902,0.881902


,offers_received,offers_viewed,view_rate,conv_rate
count,16994.000,16994.000,16994.00,16994.000
mean,4.488,3.216,0.72,0.449
std,1.073,1.273,0.24,0.319
min,1.000,0.000,0.00,0.000
25%,4.000,2.000,0.50,0.200
50%,5.000,3.000,0.75,0.500
75%,5.000,4.000,1.00,0.667
max,6.000,6.000,1.00,1.000


O processed cobre 16.994 de 17.000 clientes; as `hist_*` resumem comportamento pré-envio por linha de recebimento.


## 8. Funil, conversão e recorrência de recompra

Fonte: processed. Mede recebido, visto, convertido e recompra na janela de recorrência, com denominadores explícitos.

In [8]:
from pyspark.sql import functions as F

import numpy as np

from src import eda
from src.viz import format_pt_br, grouped_bars, line_series, vertical_bars

# SPARK REDUCE
funil = eda.response_funnel(processed)
ondas = eda.campaign_waves(processed)
por_onda = (
    processed.groupBy("campaign_wave")
    .agg(
        F.count("*").alias("recebidos"),
        F.sum("treatment").alias("vistos"),
        F.sum("converted").alias("convertidos"),
    )
    .orderBy("campaign_wave")
    .toPandas()
    .assign(
        taxa_view=lambda d: d["vistos"] / d["recebidos"],
        taxa_conversao=lambda d: d["convertidos"] / d["recebidos"],
        taxa_conversao_vistos=lambda d: np.where(d["vistos"] > 0, d["convertidos"] / d["vistos"], np.nan),
    )
)
rec_onda = eda.recurrence_by_wave(processed)
rec_trat = eda.recurrence_by_treatment(processed)
rec_pivot = (
    rec_trat.assign(grupo=rec_trat["treatment"].map({0: "não viu", 1: "viu"}))
    .pivot(index="offer_type", columns="grupo", values="taxa_recorrencia_convertidos")
    .reset_index()
)

# PANDAS ANALYSIS
display(funil.round(4))
display(por_onda.round(4))
display(rec_onda.round(4))
display(rec_trat.round(4))

taxa_media_rec = funil["taxa_recorrencia_convertidos"].mean()

grouped_bars(
    funil,
    category="offer_type",
    series=["recebidos", "vistos", "convertidos", "recorrentes"],
    series_names=["recebidos", "vistos", "convertidos", f"recorrentes ({cfg.recurrence_window_days}d)"],
    title="Funil completo por tipo — recebido, visto, convertido e recorrente",
    subtitle=f"recorrentes = outra compra em até {cfg.recurrence_window_days} dias após conversão",
    ylabel="linhas",
    orientation="v",
).show()

vertical_bars(
    funil,
    category="offer_type",
    value="taxa_conversao",
    title="Taxa de conversão sobre recebidos difere da taxa sobre vistos",
    subtitle="taxa_conversao = convertidos / recebidos",
    ylabel="Taxa sobre recebidos",
    label_template="%{y:.1%}",
    tickformat=".0%",
    hovertemplate="%{x}<br>%{y:.1%}<extra></extra>",
).show()

vertical_bars(
    funil,
    category="offer_type",
    value="taxa_recorrencia_convertidos",
    title=f"{taxa_media_rec:.0%} dos convertidos, em média, recompram em {cfg.recurrence_window_days} dias",
    subtitle="taxa_recorrencia_convertidos = recorrentes / convertidos",
    ylabel="Taxa sobre convertidos",
    label_template="%{y:.1%}",
    tickformat=".0%",
    hovertemplate="%{x}<br>%{y:.1%}<extra></extra>",
).show()

line_series(
    por_onda,
    x="campaign_wave",
    y="taxa_conversao",
    title="Conversão cai nas ondas tardias — censura à direita",
    subtitle="taxa_conversao = convertidos / recebidos por onda",
    xlabel="onda de campanha",
    ylabel="taxa de conversão",
    tickformat=".0%",
    mode="lines+markers",
).show()

line_series(
    rec_onda,
    x="campaign_wave",
    y="taxa_recorrencia_convertidos",
    title="Recorrência entre convertidos é estável entre ondas",
    subtitle=f"taxa_recorrencia_convertidos = recorrentes / convertidos · janela {cfg.recurrence_window_days} dias",
    xlabel="onda de campanha",
    ylabel="taxa de recorrência",
    tickformat=".0%",
    mode="lines+markers",
).show()

grouped_bars(
    rec_pivot,
    category="offer_type",
    series=["não viu", "viu"],
    title="Recorrência entre convertidos é parecida com e sem view",
    subtitle=f"taxa_recorrencia_convertidos por braço · janela {cfg.recurrence_window_days} dias",
    ylabel="taxa sobre convertidos",
    orientation="v",
    value_label="%{y:.1%}",
).update_layout(yaxis_tickformat=".0%").show()

total_rec = int(funil["recorrentes"].sum())
total_conv = int(funil["convertidos"].sum())
print(
    f"Recorrência observacional: {format_pt_br(total_rec)} compras com outra conversão em "
    f"{cfg.recurrence_window_days} dias sobre {format_pt_br(total_conv)} convertidos "
    f"({total_rec / total_conv:.1%}). Não mede efeito causal da oferta."
)

,offer_type,recebidos,vistos,convertidos,receita,custo,taxa_view,taxa_conversao,taxa_conversao_vistos,margem_por_envio
0,bogo,30499,24514,13496,270437.44,96795.0,0.8038,0.4425,0.5505,5.6934
1,discount,30543,20262,12500,303389.07,34773.0,0.6634,0.4093,0.6169,8.7947
2,informational,15235,9878,8008,89069.99,0.0,0.6484,0.5256,0.8107,5.8464


,campaign_wave,recebidos,vistos,convertidos,taxa_view,taxa_conversao,taxa_conversao_vistos
0,0,12650,9718,5725,0.7682,0.4526,0.5891
1,1,12669,9634,6077,0.7604,0.4797,0.6308
2,2,12711,8915,6629,0.7014,0.5215,0.7436
3,3,12778,8904,5429,0.6968,0.4249,0.6097
4,4,12704,8523,5645,0.6709,0.4443,0.6623
5,5,12765,8960,4499,0.7019,0.3524,0.5021


## 9. Tratamento, controle e conversão espontânea

Fonte: processed. Compara quem viu a oferta com quem não viu para avaliar se a resposta observada deve ser tratada como problema de uplift.

In [9]:
from pyspark.sql import functions as F

import numpy as np

from src import eda
from src.viz import format_pt_br, grouped_bars, markers_with_thresholds

# SPARK REDUCE
comparacao = eda.treatment_group_comparison(processed)
balanco = eda.covariate_balance(processed, cfg)

por_tipo = (
    processed.groupBy("treatment", "offer_type")
    .agg(
        F.count("*").alias("recebidos"),
        F.sum("converted").alias("convertidos"),
        F.sum("conversion_value").alias("receita"),
        F.sum("reward_cost").alias("custo"),
    )
    .toPandas()
    .assign(taxa_conversao=lambda d: d["convertidos"] / d["recebidos"])
)

tabela_2x2 = (
    processed.groupBy("treatment", "converted")
    .agg(F.count("*").alias("linhas"))
    .toPandas()
)
pivot = (
    tabela_2x2.pivot(index="treatment", columns="converted", values="linhas")
    .fillna(0)
    .astype(int)
    .rename(columns={0: "não converteu", 1: "converteu"})
)
pivot.index = pivot.index.map({0: "não viu (controle)", 1: "viu (tratado)"})

taxa_controle = pivot.loc["não viu (controle)", "converteu"] / pivot.loc["não viu (controle)"].sum()
taxa_tratado = pivot.loc["viu (tratado)", "converteu"] / pivot.loc["viu (tratado)"].sum()

# PANDAS ANALYSIS
display(comparacao.round(4))
display(pivot)
display(por_tipo.round(4))
display(balanco.round(3))

por_tipo_wide = (
    por_tipo.pivot(index="offer_type", columns="treatment", values="taxa_conversao")
    .rename(columns={0: "não viu", 1: "viu"})
    .reset_index()
)
grouped_bars(
    por_tipo_wide,
    category="offer_type",
    series=["não viu", "viu"],
    series_names=["não viu (controle)", "viu (tratado)"],
    title="Quem viu converte mais — mas ver é escolha do cliente",
    subtitle="taxa_conversao = convertidos / recebidos, por braço e tipo",
    ylabel="taxa",
    orientation="v",
    value_label="%{y:.1%}",
).show()

desbalanceadas = balanco.loc[balanco["acima_do_limiar"], "covariavel"].tolist()
n_acima = len(desbalanceadas)

comparavel = balanco[np.isfinite(balanco["smd"])].copy()
comparavel["índice_controle"] = 100.0
comparavel["índice_tratado"] = np.where(
    comparavel["media_controle"].abs() > 1e-9,
    100 * comparavel["media_tratado"] / comparavel["media_controle"],
    np.where(comparavel["media_tratado"].abs() < 1e-9, 100.0, np.nan),
)
comparavel = comparavel.dropna(subset=["índice_tratado"]).sort_values("abs_smd")

titulo_indice = (
    f"Com hist_*, {n_acima} covariáveis separam quem viu de quem não viu"
    if n_acima
    else "Perfil e hist_* quase iguais entre braços"
)
grouped_bars(
    comparavel,
    category="covariavel",
    series=["índice_controle", "índice_tratado"],
    series_names=["não viu (controle)", "viu (tratado)"],
    title=titulo_indice,
    subtitle=(
        f"índice com controle = 100 · inclui {len(eda.HIST_BALANCE_FEATURES)} features hist_* · "
        f"limiar |SMD| = {cfg.smd_threshold}"
    ),
    orientation="h",
    value_label="%{x:.0f}",
    xlabel="índice (controle = 100)",
).show()

titulo_balanco = (
    f"Ver a oferta não é aleatório: {n_acima} covariável(is) acima de |SMD| {cfg.smd_threshold}"
    if n_acima
    else f"Tratado e controle são comparáveis: nenhuma covariável acima de |SMD| {cfg.smd_threshold}"
)

markers_with_thresholds(
    comparavel,
    value="smd",
    label="covariavel",
    flag="acima_do_limiar",
    thresholds=(-cfg.smd_threshold, cfg.smd_threshold),
    title=titulo_balanco,
    subtitle=(
        "SMD em perfil + hist_* + gênero — quem abre a oferta já tinha mais histórico "
        "de ofertas e compras; diagnóstico, não gate"
    ),
    xlabel=f"SMD (linhas pontilhadas: ±{cfg.smd_threshold})",
).show()

print(
    f"Conversão no controle (não viu): {taxa_controle:.1%}; no tratado (viu): {taxa_tratado:.1%}. "
    "μ₀ > 0 torna o uplift estimável — ver é tratamento, não rótulo (G3)."
)
print(
    f"Covariáveis acima do limiar SMD={cfg.smd_threshold}: "
    f"{n_acima} ({', '.join(desbalanceadas) if desbalanceadas else 'nenhuma'}). "
    "O desbalanço aparece nas hist_* de engajamento — quem viu já recebia/comprava mais; "
    "o perfil demográfico sozinho ainda parece equilibrado."
)

,treatment,recebidos,taxa_conversao,ticket_medio,receita_media
0,não viu,21623,0.3302,19.6469,6.4875
1,viu,54654,0.4915,19.4542,9.5623


converted,não converteu,converteu
treatment,,
não viu (controle),14483,7140
viu (tratado),27790,26864


,treatment,offer_type,recebidos,convertidos,receita,custo,taxa_conversao
0,1,bogo,24514,11228,227429.21,82505.0,0.4580
1,0,discount,10281,2532,71774.63,8421.0,0.2463
2,1,informational,9878,5668,63573.98,0.0,0.5738
3,1,discount,20262,9968,231614.44,26352.0,0.4920
4,0,informational,5357,2340,25496.01,0.0,0.4368
5,0,bogo,5985,2268,43008.23,14290.0,0.3789


,covariavel,media_tratado,media_controle,smd,abs_smd,acima_do_limiar
0,hist_offers_received,1.808,2.034,-0.156,0.156,True
1,hist_offers_received_discount,0.718,0.833,-0.127,0.127,True
2,hist_txn_count,3.331,3.774,-0.120,0.120,True
3,hist_offers_received_bogo,0.721,0.813,-0.104,0.104,True
4,hist_frequency,0.198,0.217,-0.097,0.097,False
5,hist_completed_unseen_flag,0.142,0.176,-0.092,0.092,False
6,identity_missing,0.136,0.109,0.082,0.082,False
7,gender=unknown,0.136,0.109,0.082,0.082,False
8,credit_card_limit,65770.918,64392.859,0.063,0.063,False
9,gender=M,0.492,0.521,-0.059,0.059,False


Conversão no controle (não viu): 33.0%; no tratado (viu): 49.2%. μ₀ > 0 torna o uplift estimável — ver é tratamento, não rótulo (G3).
Covariáveis acima do limiar SMD=0.1: 4 (hist_offers_received, hist_offers_received_discount, hist_txn_count, hist_offers_received_bogo).


## 10. Economia observada das ofertas

Fonte: processed. Examina valor de conversão, custo de recompensa e ticket para conectar resposta estatística com impacto de negócio.

In [10]:
from pyspark.sql import functions as F

from src.viz import format_pt_br, grouped_bars

# SPARK REDUCE
economia = (
    processed.groupBy("offer_type", "treatment")
    .agg(
        F.count("*").alias("recebidos"),
        F.sum("converted").alias("convertidos"),
        F.sum("conversion_value").alias("receita"),
        F.sum("reward_cost").alias("custo"),
    )
    .orderBy("offer_type", "treatment")
    .toPandas()
)
economia["margem"] = economia["receita"] - economia["custo"]
economia["margem_por_envio"] = economia["margem"] / economia["recebidos"]
economia["ticket_medio"] = np.where(
    economia["convertidos"] > 0,
    economia["receita"] / economia["convertidos"],
    np.nan,
)

por_onda = (
    processed.groupBy("campaign_wave", "offer_type")
    .agg(
        F.count("*").alias("recebidos"),
        F.sum("conversion_value").alias("receita"),
        F.sum("reward_cost").alias("custo"),
    )
    .toPandas()
    .assign(margem_por_envio=lambda d: (d["receita"] - d["custo"]) / d["recebidos"])
    .sort_values(["offer_type", "campaign_wave"])
)

# PANDAS ANALYSIS
display(economia.round(2))
display(por_onda.round(2))

tratado = economia[economia["treatment"] == 1].copy()
grouped_bars(
    tratado,
    category="offer_type",
    series=["receita", "custo", "margem"],
    series_names=["receita", "custo", "margem líquida"],
    title="Discount gera mais margem por envio que bogo no braço exposto",
    subtitle="somente treatment=1 (viu) · valores em R$",
    ylabel="R$",
    orientation="v",
).show()

grouped_bars(
    tratado,
    category="offer_type",
    series=["margem_por_envio"],
    series_names=["margem / envio"],
    title="Lucro observado por envio não coincide com taxa de conversão",
    subtitle="margem_por_envio = (receita − custo) / recebidos",
    ylabel="R$ por envio",
    orientation="v",
).show()

print(
    "Alta conversão e alto valor por envio não apontam para a mesma oferta: "
    "bogo converte mais, discount rende mais por envio no tratado."
)

,offer_type,treatment,recebidos,convertidos,receita,custo,margem,margem_por_envio,ticket_medio
0,bogo,0,5985,2268,43008.23,14290.0,28718.23,4.80,18.96
1,bogo,1,24514,11228,227429.21,82505.0,144924.21,5.91,20.26
2,discount,0,10281,2532,71774.63,8421.0,63353.63,6.16,28.35
3,discount,1,20262,9968,231614.44,26352.0,205262.44,10.13,23.24
4,informational,0,5357,2340,25496.01,0.0,25496.01,4.76,10.90
5,informational,1,9878,5668,63573.98,0.0,63573.98,6.44,11.22


,campaign_wave,offer_type,recebidos,receita,custo,margem_por_envio
17,0,bogo,5018,47463.84,16520.0,6.17
4,1,bogo,5118,50218.66,17910.0,6.31
2,2,bogo,5047,53834.34,18815.0,6.94
13,3,bogo,5110,39079.56,14755.0,4.76
7,4,bogo,5124,43098.07,15995.0,5.29
14,5,bogo,5082,36742.97,12800.0,4.71
5,0,discount,5093,51812.90,6015.0,8.99
6,1,discount,5015,54269.27,6114.0,9.60
10,2,discount,5129,61874.56,6990.0,10.70
1,3,discount,5100,50198.28,5815.0,8.70


Alta conversão e alto valor por envio não apontam para a mesma oferta: bogo converte mais, discount rende mais por envio no tratado.


## 11. Síntese para modelagem

Fonte: raw e processed. Resume os achados que devem guiar a modelagem de qual oferta enviar para cada cliente.

In [11]:
from src.viz import format_pt_br

# PANDAS ANALYSIS — síntese dos achados que guiam a modelagem de uplift

sintese = pd.DataFrame([
    {
        "achado": "Grão condensado e disparos discretos",
        "evidência": f"{format_pt_br(events.count())} eventos → {format_pt_br(processed.count())} recebimentos em {len(ondas)} ondas",
        "implicação_modelagem": "Features as-of received_time; campaign_wave captura timing, não contínuo",
        "implicação_negócio": "Budget e validade seguem ondas, não calendário contínuo",
    },
    {
        "achado": "Controle converte sem ver (G3)",
        "evidência": f"taxa controle {taxa_controle:.1%} vs tratado {taxa_tratado:.1%}; completados sem view {taxa_sem_view:.1%}",
        "implicação_modelagem": "X-learner com μ₀ > 0; label ≠ exposição",
        "implicação_negócio": "Oferta move quem já compraria e quem precisa do empurrão",
    },
    {
        "achado": "Mix bogo/discount/informational",
        "evidência": "10 ofertas · funil e margem diferem por tipo",
        "implicação_modelagem": "Estratificar por offer_type; informational fora da modelagem",
        "implicação_negócio": "Decisão é qual cupom enviar, não só se enviar",
    },
    {
        "achado": "Features históricas informativas",
        "evidência": "hist_spend_total, hist_view_rate e hist_time_view_to_conv com cauda e correlação",
        "implicação_modelagem": "hist_* entram no X-learner; recência e taxas capturam hábito pré-envio",
        "implicação_negócio": "Quem já compra/view diferente responde diferente à oferta",
    },
    {
        "achado": "Economia ≠ taxa de conversão",
        "evidência": "discount: maior margem/envio; bogo: maior conversão",
        "implicação_modelagem": "Score de ranking precisa ponderar τ e ticket/custo",
        "implicação_negócio": "Otimizar lucro líquido, não só conversão",
    },
    {
        "achado": "Censura e desbalanço pós-view",
        "evidência": (
            f"ondas tardias com conversão menor; "
            f"{len(desbalanceadas)} hist_* acima de |SMD| {cfg.smd_threshold} "
            f"({', '.join(desbalanceadas[:3])}{'…' if len(desbalanceadas) > 3 else ''})"
        ),
        "implicação_modelagem": (
            "Holdout temporal; uplift sob ignorabilidade condicional às features — "
            "não por aleatorização do view"
        ),
        "implicação_negócio": "Efeito das últimas ondas é subestimado; quem vê já engajava mais",
    },
])

display(sintese)

print(
    "Pergunta de modelagem: para cada cliente, qual oferta (bogo ou discount) "
    "maximiza o lucro líquido incremental — τ de conversão menos custo de recompensa?"
)

,achado,evidência,implicação_modelagem,implicação_negócio
0,Grão condensado e disparos discretos,306.534 eventos → 76.277 recebimentos em 6 ondas,Features as-of received_time; campaign_wave ca...,"Budget e validade seguem ondas, não calendário..."
1,Controle converte sem ver (G3),taxa controle 33.0% vs tratado 49.2%; completa...,X-learner com μ₀ > 0; label ≠ exposição,Oferta move quem já compraria e quem precisa d...
2,Mix bogo/discount/informational,10 ofertas · funil e margem diferem por tipo,Estratificar por offer_type; informational for...,"Decisão é qual cupom enviar, não só se enviar"
3,Features históricas informativas,"hist_spend_total, hist_view_rate e hist_time_v...",hist_* entram no X-learner; recência e taxas c...,Quem já compra/view diferente responde diferen...
4,Economia ≠ taxa de conversão,discount: maior margem/envio; bogo: maior conv...,Score de ranking precisa ponderar τ e ticket/c...,"Otimizar lucro líquido, não só conversão"
5,Censura e balanço,ondas tardias com conversão menor; SMD < 0.1 e...,Holdout temporal; ignorabilidade plausível mas...,Efeito das últimas ondas é subestimado


Pergunta de modelagem: para cada cliente, qual oferta (bogo ou discount) maximiza o lucro líquido incremental — τ de conversão menos custo de recompensa?


## Teardown

Encerra a sessão Spark.

In [12]:
# SPARK TEARDOWN
spark.stop()
print("Sessão Spark encerrada.")

Sessão Spark encerrada.
